# 🔧 FER2013 Pre-training (Tùy chọn)

Notebook này dùng để **pretrain trên FER2013** trước, sau đó dùng checkpoint đó làm khởi điểm cho DAiSEE.

**Khi nào nên dùng?** Khi DAiSEE training cho kết quả chưa tốt (val_accuracy < 55%).

> **Lợi ích:** FER2013 có 35,887 ảnh khuôn mặt đã được crop sẵn → model học được facial features trước, sau đó DAiSEE fine-tune thêm context lớp học.

In [ ]:
    # CELL 1 — Cài thư viện & Kết nối Kaggle
    !pip install -q kaggle

    from google.colab import files
    print('Upload kaggle.json:')
    uploaded = files.upload()
    !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

    # Tải FER2013 (chỉ ~60MB, nhanh hơn nhiều so với DAiSEE)
    !kaggle datasets download -d msambare/fer2013 -p /content/fer2013 --unzip
    print('✅ FER2013 đã tải!')

    import os
    for root, dirs, files_list in os.walk('/content/fer2013'):
        level = root.replace('/content/fer2013', '').count(os.sep)
        indent = '  ' * level
        print(f'{indent}{os.path.basename(root)}/')
        if level < 2:
            for f in files_list[:3]: print(f'{indent}  {f}')

In [ ]:
# CELL 2 — Ánh xạ nhãn FER2013 → 4 nhãn hệ thống
import shutil
from pathlib import Path
import os

# FER2013 gốc có 7 nhãn:
# angry, disgust, fear, happy, sad, surprise, neutral
FER_MAPPING = {
    'angry':    0,  # Buồn chán / tiêu cực
    'disgust':  0,  # Buồn chán
    'fear':     0,  # Buồn chán
    'sad':      0,  # Buồn chán
    'neutral':  3,  # Bình thường
    'happy':    2,  # Hứng thú
    'surprise': 2,  # Hứng thú
    # NOTE: 'Tập trung' (1) không có class tương đương trong FER2013
    # → Sẽ fine-tune thêm với DAiSEE
}

EMOTION_NAMES = ['Buon_chan', 'Tap_trung', 'Hung_thu', 'Binh_thuong']
OUTPUT = Path('/content/fer_mapped')

for split in ['train', 'test']:
    for i in range(4):
        (OUTPUT / split / str(i)).mkdir(parents=True, exist_ok=True)

fer_base = Path('/content/fer2013')
# Tìm thư mục train/test trong FER2013
splits = {}
for p in fer_base.rglob('*'):
    if p.is_dir() and p.name.lower() in ['train', 'training']:
        splits['train'] = p
    elif p.is_dir() and p.name.lower() in ['test', 'validation', 'val']:
        splits['test'] = p

count = 0
for split_name, split_path in splits.items():
    for cls_dir in split_path.iterdir():
        if not cls_dir.is_dir(): continue
        fer_label = cls_dir.name.lower()
        new_label = FER_MAPPING.get(fer_label, 3)
        for img_path in cls_dir.glob('*.jpg'):
            dst = OUTPUT / split_name / str(new_label) / img_path.name
            shutil.copy2(img_path, dst)
            count += 1
        for img_path in cls_dir.glob('*.png'):
            dst = OUTPUT / split_name / str(new_label) / img_path.name
            shutil.copy2(img_path, dst)
            count += 1

print(f'✅ Đã ánh xạ {count:,} ảnh FER2013!')
for split in ['train', 'test']:
    print(f'  {split}:')
    for i, name in enumerate(EMOTION_NAMES):
        n = len(list((OUTPUT/split/str(i)).glob('*.jpg'))) + len(list((OUTPUT/split/str(i)).glob('*.png')))
        print(f'    Class {i} ({name}): {n:,} ảnh')

In [ ]:
# CELL 3 — Train nhanh trên FER2013 (10 epochs)
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
import numpy as np

IMG_SIZE   = (224, 224)
BATCH_SIZE = 64
NUM_CLASSES= 4

train_gen = ImageDataGenerator(rescale=1./255, horizontal_flip=True,
                                rotation_range=10, zoom_range=0.1)
val_gen   = ImageDataGenerator(rescale=1./255)

train_flow = train_gen.flow_from_directory(
    str(OUTPUT / 'train'), target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode='categorical')

val_flow = val_gen.flow_from_directory(
    str(OUTPUT / 'test'), target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False)

# Build model
base = MobileNetV2(input_shape=(*IMG_SIZE,3), include_top=False, weights='imagenet')
base.trainable = False
x = layers.GlobalAveragePooling2D()(base.output)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.4)(x)
out = layers.Dense(NUM_CLASSES, activation='softmax')(x)
model = Model(base.input, out)

cw_arr = compute_class_weight('balanced', classes=np.unique(train_flow.classes), y=train_flow.classes)
cw_dict = dict(enumerate(cw_arr))

model.compile(Adam(1e-3), 'categorical_crossentropy', ['accuracy'])

os.makedirs('/content/checkpoints', exist_ok=True)
history = model.fit(
    train_flow, validation_data=val_flow, epochs=10,
    class_weight=cw_dict,
    callbacks=[
        ModelCheckpoint('/content/checkpoints/fer_pretrain.h5',
                        monitor='val_accuracy', save_best_only=True, mode='max')
    ]
)

best_acc = max(history.history['val_accuracy'])
print(f'\n✅ FER2013 Pretrain xong! Best val_acc = {best_acc:.4f}')
print('   Tiếp tục mở file 01_DAiSEE_Training.ipynb')
print('   Thay MODEL_PATH = "/content/checkpoints/fer_pretrain.h5" ở Cell 7')